# Phase 2 comparison -- June-Aug (unfiltered/filtered) vs. Sept-Nov (filtered)

**Run this only after `code_phase2_harbin_unfiltered.ipynb`, `code_phase2_harbin_filtered.ipynb`, and `code_phase2_harbin_sepnov.ipynb` have all finished training and saved their models to Drive.** This notebook is fully self-contained (its own Drive mount, its own extraction of all three datasets, its own model loading) -- it doesn't depend on any training notebook's session state, so it's safe to run in a completely fresh Colab runtime.

**What's being compared, and why the statistical test differs between the two comparisons here** -- this notebook makes two genuinely different kinds of comparison, and treats them differently on purpose:

1. **Unfiltered vs. filtered (June-Aug)**: both share the identical 917/211/184 train/val/test split (same underlying images, same acquisition dates, only the mask-generation rule differs). Every validation image has an unfiltered-model prediction *and* a filtered-model prediction on the exact same input -- a genuine paired design, so the paired Wilcoxon signed-rank test used in `code_phase2_harbin_comparison.ipynb` is appropriate here too (reproduced below for completeness).

2. **Filtered June-Aug vs. Sept-Nov**: these come from *different acquisition windows* -- different dates, and not even guaranteed to be the same tile positions after the independent 70/15/15 hash-split each dataset went through. There is no shared validation image to pair predictions on, so treating this like (1) with a paired test would be invalid (Wilcoxon assumes the two samples are matched pairs of the same underlying unit). Instead this uses an **unpaired Mann-Whitney U test** (`scipy.stats.mannwhitneyu`) on the two independent per-image accuracy distributions -- non-parametric for the same reason as before (per-image accuracy is a bounded, skewed proportion, not normally distributed), but appropriate for two independent samples rather than matched pairs.

In [ ]:
import os
# Force legacy Keras 2 behaviour. Current TF ships Keras 3 by default, which
# breaks this codebase's private optimizer-internals usage and the
# unmaintained `segmentation_models` package -- tf_keras is TF's official
# compatibility shim for exactly this situation, same fix already applied
# in predictor.py.
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import sys
import numpy as np
import pandas as pd
import PIL
import requests
import tensorflow as tf
import tf_keras as keras
from tf_keras.models import *
from tf_keras.layers import *
from tf_keras.optimizers import *
from tf_keras.losses import *
from tf_keras import backend as K
from tf_keras.callbacks import ModelCheckpoint
from tf_keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import *


In [ ]:
# Colab setup: mount Drive, clone the repo, extract all THREE ground-truth
# archives, and locate all three trained models (copied to Drive at the
# end of each training notebook).
import os
import shutil

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

VERSIONS = ("unfiltered", "filtered", "sepnov")
MODEL_FILENAMES = {
    "unfiltered": "unet-attention-4d-harbin-unfiltered.hdf5",
    "filtered": "unet-attention-4d-harbin-filtered.hdf5",
    "sepnov": "unet-attention-4d-harbin-sepnov.hdf5",
}

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = "/content/COMP0173_poster_pre"
    if not os.path.exists(REPO_DIR):
        !git clone -q https://github.com/jy-gfm/COMP0173_poster_pre.git {REPO_DIR}
    os.chdir(REPO_DIR)

    DRIVE_DIR = "/content/drive/MyDrive/COMP0173/"
    DATA_DIRS = {}
    MODEL_PATHS = {}

    for version in VERSIONS:
        tar_name = f"Haerbing_ground_truth_{version}"
        local_data_dir = f"/content/{tar_name}"
        if not os.path.exists(local_data_dir):
            extract_root = f"/content/{tar_name}_extract"
            os.makedirs(extract_root, exist_ok=True)
            !tar -xf {DRIVE_DIR}{tar_name}.tar -C {extract_root} 2>&1 | grep -v "Ignoring unknown extended header keyword" || true
            local_data_dir = f"{extract_root}/{tar_name}"
        DATA_DIRS[version] = local_data_dir

        local_model_path = f"/content/{MODEL_FILENAMES[version]}"
        if not os.path.exists(local_model_path):
            shutil.copy(f"{DRIVE_DIR}{MODEL_FILENAMES[version]}", local_model_path)
        MODEL_PATHS[version] = local_model_path
else:
    DATA_DIRS = {v: f"./Haerbing_ground_truth_{v}" for v in VERSIONS}
    MODEL_PATHS = {v: MODEL_FILENAMES[v] for v in VERSIONS}

print("DATA_DIRS:", DATA_DIRS)
print("MODEL_PATHS:", MODEL_PATHS)


## Load all three models and compute per-image validation accuracy for each

In [ ]:
import numpy as np
import glob
from scipy.stats import wilcoxon, mannwhitneyu

def load_validation(version):
    paths = sorted(glob.glob(f"{DATA_DIRS[version]}/validation/images/*.npy"))
    images = [np.load(p).reshape(1, 512, 512, 4) for p in paths]
    masks = [np.load(p.replace("/images/", "/masks/")).reshape(1, 512, 512, 1) for p in paths]
    return paths, images, masks

def per_image_accuracy(model, images, masks):
    scores = []
    for img, mask in zip(images, masks):
        pred = np.round(model.predict(img, verbose=0)).flatten()
        scores.append((pred == mask.flatten()).mean())
    return np.array(scores)

models = {v: keras.models.load_model(MODEL_PATHS[v], compile=False) for v in VERSIONS}

paths, images, masks, acc = {}, {}, {}, {}
for v in VERSIONS:
    paths[v], images[v], masks[v] = load_validation(v)
    acc[v] = per_image_accuracy(models[v], images[v], masks[v])
    print(f"{v}: n={len(acc[v])}, mean per-image accuracy = {acc[v].mean():.4f} (+/- {acc[v].std():.4f})")


## Comparison 1 (paired) -- unfiltered vs. filtered, June-Aug

In [ ]:
# Sanity check: same underlying positions in the same order in both datasets
assert [os.path.basename(p) for p in paths["unfiltered"]] == [os.path.basename(p) for p in paths["filtered"]], \
    "unfiltered/filtered validation sets don't line up -- can't pair them"

stat_uf, p_uf = wilcoxon(acc["unfiltered"], acc["filtered"])
print(f"Wilcoxon signed-rank test (unfiltered vs. filtered): statistic={stat_uf:.2f}, p-value={p_uf:.4g}")
print("Statistically significant at alpha=0.05." if p_uf < 0.05 else "Not statistically significant at alpha=0.05.")


## Comparison 2 (unpaired) -- filtered June-Aug vs. Sept-Nov

Different images, different acquisition dates, independently split -- so this uses the unpaired Mann-Whitney U test on the two accuracy distributions rather than pairing by position (see this notebook's opening cell for why).

In [ ]:
stat_season, p_season = mannwhitneyu(acc["filtered"], acc["sepnov"], alternative="two-sided")
print(f"Mann-Whitney U test (filtered June-Aug vs. Sept-Nov): statistic={stat_season:.2f}, p-value={p_season:.4g}")
print("Statistically significant at alpha=0.05." if p_season < 0.05 else "Not statistically significant at alpha=0.05.")

better_season = "Sept-Nov" if acc["sepnov"].mean() > acc["filtered"].mean() else "June-Aug"
print(f"\nHigher mean per-image accuracy: {better_season} "
      f"(June-Aug filtered={acc['filtered'].mean():.4f}, Sept-Nov={acc['sepnov'].mean():.4f})")


## Visualize all three

In [ ]:
import matplotlib.pyplot as plt

labels = ["unfiltered\n(Jun-Aug)", "filtered\n(Jun-Aug)", "sepnov\n(filtered)"]
colors = ["#8899AA", "#3B7A57", "#C9A227"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Mean accuracy with std error bars, all three models
means = [acc["unfiltered"].mean(), acc["filtered"].mean(), acc["sepnov"].mean()]
stds = [acc["unfiltered"].std(), acc["filtered"].std(), acc["sepnov"].std()]
axes[0].bar(labels, means, yerr=stds, capsize=8, color=colors)
axes[0].set_ylabel("mean per-image accuracy")
axes[0].set_title("Mean accuracy (error bars = std)")
axes[0].set_ylim(0, 1)

# 2. Full distribution of per-image accuracy, all three models
axes[1].boxplot([acc["unfiltered"], acc["filtered"], acc["sepnov"]], labels=labels)
axes[1].set_ylabel("per-image accuracy")
axes[1].set_title("Distribution of per-image accuracy")

# 3. Paired scatter -- ONLY for the genuinely paired comparison
# (unfiltered vs. filtered, same validation images). No equivalent plot
# for sepnov since it has no shared images to pair against.
lims = [0, 1]
axes[2].plot(lims, lims, 'k--', linewidth=1, label="y = x")
axes[2].scatter(acc["unfiltered"], acc["filtered"], alpha=0.5, s=15)
axes[2].set_xlabel("unfiltered accuracy (Jun-Aug)")
axes[2].set_ylabel("filtered accuracy (Jun-Aug)")
axes[2].set_title(f"Paired per-image comparison\n(Wilcoxon p={p_uf:.4g})")
axes[2].legend()
axes[2].set_xlim(lims)
axes[2].set_ylim(lims)

plt.tight_layout()
plt.show()

for v in VERSIONS:
    print(f"{v}: {acc[v].mean():.4f} +/- {acc[v].std():.4f} (n={len(acc[v])})")
